# Compact CNN: saved EEG trials
Use **Runtime → Change runtime type → T4 GPU**, then **Run all**.

One independent EEGNet-inspired model; no tuning search. Positive/negative and neutral/nonneutral, separately for ZuCo SR/NR/TSR and TeCo task 1. Three grouped folds each for held-out sentences, words and readers (72 fold trainings total). Class weights retain every neutral trial. Inner grouped validation selects the stopping epoch; test trials never select settings.

Reads the existing 40 Hz/125 Hz all-channel trial archive. Input: 0–792 ms, all channels × 100 samples. **No extraction or old feature preparation.** Reading archive shards into temporary Colab storage takes time once per task/runtime. Up to roughly 3 GB temporary cloud disk, removed per completed task; **no second EEG copy in Drive**. A disconnected runtime may need to read unfinished tasks again. Completed tasks/folds are reused.

Additional Drive outputs: **estimated 20–60 MB**, with a 150 MB guard. Only small models, learning curves, splits, predictions and scores. Training may take hours; epoch checkpoints are saved. Original results stay intact. This is an additional classifier, not the paper reproduction.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, importlib.util
THESIS=Path('/content/drive/MyDrive/Thesis')
REPO=Path('/content/word-sentiment-eeg-cnn')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/parmisbathayan/word-sentiment-eeg.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
missing=[name for name in ['numpy','pandas','scipy','sklearn','matplotlib'] if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-analysis.txt')],check=True)
if importlib.util.find_spec('torch') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','torch'],check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime and rerun. No analysis has started.')
print('GPU:',torch.cuda.get_device_name())
print('Code:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())
OUT=THESIS/'Results/word_sentiment_eeg/cnn_eegnet'


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'scripts/classify_cnn.py'),'--thesis-root',str(THESIS)]
run_with_progress(command,OUT/'runtime_status.json',activity='CNN using saved individual EEG trials')


## Classification results
Balanced accuracy gives each class equal weight. Chance is 50%. Each holdout tests a different form of generalization; the three restrictions are not simultaneous.

In [ ]:
import pandas as pd
from IPython.display import display,Markdown
scores=pd.read_csv(OUT/'scores.csv')
for target in ['positive_negative','neutral_nonneutral']:
    display(Markdown('### '+target.replace('_',' ')))
    table=scores[scores.target.eq(target)].pivot(index=['dataset','task'],columns='split',values='balanced_accuracy')
    display((100*table).round(2))
print((OUT/'manifest.json').read_text())
print('Saved in:',OUT)


## CNN versus existing linear models
Same label policy and outer holdout types; CNN uses all channels and time samples. Its inner validation reserves part of training data. No claim of significance from a small score difference.

In [ ]:
linear=THESIS/'Results/word_sentiment_eeg/classification_similarity'
rows=[]
old=pd.read_csv(linear/'scores.csv')
part=old[(old.policy=='word_agreement')&(old.target=='binary')&(old.feature=='post')&(old.model=='logistic')].copy()
part['target']='positive_negative';rows.append(part)
follow=linear/'neutral_nonneutral/scores.csv'
if follow.exists():
    part=pd.read_csv(follow);part=part[(part.feature=='post')&(part.model=='logistic')].copy()
    part['target']='neutral_nonneutral';rows.append(part)
baseline=pd.concat(rows,ignore_index=True)
comparison=scores.merge(baseline[['dataset','task','target','split','balanced_accuracy']],on=['dataset','task','target','split'],suffixes=('_cnn','_logistic'))
display(comparison[['dataset','task','target','split','balanced_accuracy_cnn','balanced_accuracy_logistic']])
import matplotlib.pyplot as plt
for target,part in comparison.groupby('target'):
    fig,ax=plt.subplots(figsize=(11,4))
    labels=(part.dataset+'/'+part.task+'/'+part['split']).tolist()
    ax.plot(labels,100*part.balanced_accuracy_cnn,'o-',label='CNN')
    ax.plot(labels,100*part.balanced_accuracy_logistic,'o-',label='Logistic')
    ax.axhline(50,color='gray',linestyle='--');ax.set_ylabel('Balanced accuracy (%)');ax.set_title(target.replace('_',' '))
    ax.tick_params(axis='x',rotation=60);ax.legend();fig.tight_layout();display(fig);plt.close(fig)


## Training curves
Validation curves select the saved model. Training progress is not evidence of successful sentiment classification; use the held-out scores above.

In [ ]:
for dataset,task in [('zuco','SR'),('zuco','NR'),('zuco','TSR'),('teco','task1')]:
    fig,axes=plt.subplots(2,3,figsize=(13,6),squeeze=False)
    for row,target in enumerate(['positive_negative','neutral_nonneutral']):
        for col,mode in enumerate(['sentence','word','reader']):
            ax=axes[row,col]
            for path in sorted((OUT/dataset/task/target/mode).glob('fold_*/learning_curve.csv')):
                curve=pd.read_csv(path);ax.plot(curve.epoch,100*curve.validation_balanced_accuracy,label=path.parent.name)
            ax.axhline(50,color='gray',linestyle='--');ax.set_title(target+' / '+mode);ax.set_xlabel('Epoch');ax.set_ylabel('Validation BA (%)')
            if ax.lines:ax.legend(fontsize=7)
    fig.suptitle(dataset+' / '+task);fig.tight_layout();display(fig);plt.close(fig)
